# Arboretum Tree Heights & Crown Diameters from Tree Coordinates

Estimates height and crown diameter for each surveyed tree by combining its coordinates ("Biomass Calcs" sheet `Lat`/`Long`) with the clipped 0.25m CHM.

**Method**
1. **Height** - maximum CHM value within `search_radius` of each point (absorbs GPS error; the point rarely lands exactly on the apex).
2. **Snap** - the seed is moved to the location of that maximum, i.e. the true treetop.
3. **Crown** - seeded region growing (`lidR::dalponte2016`) from the snapped treetops gives one crown per tree.
4. **Crown diameter** - equivalent-area diameter `2*sqrt(A/pi)` and maximum Feret diameter (longest hull span).
5. **QC** - compare to `GT Height` and flag points that share a crown.

**Outputs** (written to `../outputs/`): `arboretum_tree_metrics_025m.csv`, `arboretum_treetops_025m.gpkg`, `arboretum_crowns_025m.gpkg`

## 1. Load Packages

In [ ]:
# install.packages(c('terra', 'sf', 'lidR', 'dplyr', 'readxl'), repos = 'https://cran.r-project.org')
suppressPackageStartupMessages({
  library(terra)
  library(sf)
  library(lidR)
  library(dplyr)
  library(readxl)
})

## 2. Configuration

- `smooth_for_height` - extract heights from a smoothed CHM (default `FALSE`: smoothing lowers peaks)
- `smooth_for_crowns` - smooth the CHM before crown delineation (default `TRUE`: removes pits that fragment crowns)
- `kernel_size` - odd integer, size of the mean-filter window in cells (3 = 0.75 m at 0.25 m resolution)
- `search_radius` - metres around each point searched for the treetop
- `th_tree` - minimum height (m) for a cell to belong to a tree
- `th_seed` / `th_cr` - Dalponte thresholds: a neighbour joins a crown if it is >= `th_seed` x seed height and >= `th_cr` x mean crown height
- `max_cr` - maximum crown radius in **cells** (40 cells = 10 m at 0.25 m)

In [ ]:
base <- "C:/Users/davisk10/OneDrive - Cal Poly/Tree Biomass Estimation Research - Documents"
chm_path      <- file.path(base, "GitHub_Repository/Rasters/arb_chm_0.25m_clipped.tif")
workbook_path <- file.path(base, "GitHub_Repository/Biomass_Monitoring_Research/data/WorkingTrees_Arboretum_Data.xlsx")
output_dir    <- file.path(base, "GitHub_Repository/Biomass_Monitoring_Research/outputs")

smooth_for_height <- FALSE
smooth_for_crowns <- TRUE
kernel_size       <- 3
search_radius     <- 2.0
th_tree <- 2.0
th_seed <- 0.45
th_cr   <- 0.55
max_cr  <- 40

## 3. Load CHM and Tree Coordinates

In [ ]:
chm <- rast(chm_path)
names(chm) <- "height"

# defaults = raw CHM; section 4 (optional) overrides these when smoothing is enabled
chm_height <- chm
chm_crown  <- chm
cat("CHM:", paste(dim(chm)[1:2], collapse = " x "), "cells, res", paste(res(chm), collapse = " x "), "m\n")

trees <- read_excel(workbook_path, sheet = "Biomass Calcs")
# common names sit in column H, which has no header in the sheet
names(trees)[8] <- "common_name"
trees <- trees %>%
  filter(!is.na(Lat), !is.na(Long)) %>%
  mutate(point_id = row_number())
cat("Trees with coordinates:", nrow(trees), "\n")

pts <- st_as_sf(trees, coords = c("Long", "Lat"), crs = 4326, remove = FALSE) %>%
  st_transform(crs(chm))

outside <- is.na(extract(chm, vect(pts))[, 2])
cat("Points outside CHM extent / on NoData:", sum(outside), "\n")

## 4. Smooth CHM (optional)

Skip this cell to use the raw CHM for everything (section 3 already set `chm_height` and `chm_crown`). Run it to apply the `smooth_for_*` toggles from the config.

In [ ]:
smooth_chm <- function(r, k) {
  focal(r, w = matrix(1, k, k), fun = "mean", na.policy = "omit", na.rm = TRUE)
}
if (smooth_for_height || smooth_for_crowns) {
  chm_smooth <- smooth_chm(chm, kernel_size)
  names(chm_smooth) <- "height"
  if (smooth_for_height) chm_height <- chm_smooth
  if (smooth_for_crowns) chm_crown  <- chm_smooth
}

## 5. Height and Treetop Snapping

For each point, find the highest cell within `search_radius`; that cell becomes the treetop.

In [ ]:
find_treetop <- function(pt, r, radius) {
  buf   <- vect(st_buffer(pt, radius))
  cells <- mask(crop(r, buf), buf)
  v <- values(cells)[, 1]
  if (all(is.na(v))) return(c(x = NA, y = NA, height = NA))
  i  <- which.max(v)
  xy <- xyFromCell(cells, i)
  c(x = xy[1], y = xy[2], height = v[i])
}

tops <- as.data.frame(t(sapply(seq_len(nrow(pts)), function(i) find_treetop(pts[i, ], chm_height, search_radius))))

pc <- st_coordinates(pts)
trees$chm_height_m <- tops$height
trees$top_x        <- tops$x
trees$top_y        <- tops$y
trees$snap_dist_m  <- sqrt((tops$x - pc[, 1])^2 + (tops$y - pc[, 2])^2)

summary(trees[, c("chm_height_m", "snap_dist_m")])

## 6. Crown Delineation

Dalponte seeded region growing needs unique seeds. Points that snap to the same cell (e.g. stems of one tree) are merged here and mapped back afterwards.

In [ ]:
valid <- !is.na(trees$top_x)
seeds <- trees[valid, ] %>%
  mutate(cell = cellFromXY(chm_crown, cbind(top_x, top_y))) %>%
  group_by(cell) %>%
  mutate(treeID = cur_group_id()) %>%
  ungroup()

treetops <- seeds %>%
  distinct(treeID, .keep_all = TRUE) %>%
  st_as_sf(coords = c("top_x", "top_y"), crs = crs(chm), remove = FALSE)
treetops$Z <- extract(chm_crown, vect(treetops))[, 2]
treetops <- treetops[!is.na(treetops$Z), c("treeID", "Z")]

# lidR cannot segment a raster stored on disk, so force it into memory first
chm_crown_mem <- raster::readAll(raster::raster(chm_crown))
crown_r <- dalponte2016(chm_crown_mem, treetops,
                        th_tree = th_tree, th_seed = th_seed, th_cr = th_cr, max_cr = max_cr)()
crown_r <- rast(crown_r)
names(crown_r) <- "treeID"
cat("Crowns delineated:", length(unique(na.omit(values(crown_r)))), "of", nrow(treetops), "seeds\n")

### 6b. Alternative: Marker-Controlled (Inverse) Watershed

Compares against Dalponte using the **same snapped treetops**. The CHM is treated as an inverted surface, so each treetop becomes a basin that is filled outward until it meets a neighbouring basin; the meeting lines become crown boundaries (`ForestTools::mcws`, which wraps `imager::watershed`). Unlike Dalponte there are no height-ratio thresholds, so crowns tend to expand to fill all space above `th_tree`. Results are stored in `crown_r_ws` / `crowns_ws` and do not affect the Dalponte outputs below.

In [ ]:
library(ForestTools)

ws_r <- mcws(treetops = treetops, CHM = chm_crown, minHeight = th_tree, IDfield = "treeID", format = "raster")
crown_r_ws <- ws_r
names(crown_r_ws) <- "treeID"

crowns_ws <- st_as_sf(as.polygons(crown_r_ws, dissolve = TRUE))
crowns_ws <- crowns_ws[!is.na(crowns_ws$treeID), ]
crowns_ws$crown_area_m2   <- as.numeric(st_area(crowns_ws))
crowns_ws$crown_diam_eq_m <- 2 * sqrt(crowns_ws$crown_area_m2 / pi)
cat("Watershed crowns:", nrow(crowns_ws), "of", nrow(treetops), "seeds
")

# per-tree comparison with Dalponte (same treeID = same seed)
cmp_ws <- merge(st_drop_geometry(crowns)[, c("treeID", "crown_area_m2", "crown_diam_eq_m")],
                st_drop_geometry(crowns_ws)[, c("treeID", "crown_area_m2", "crown_diam_eq_m")],
                by = "treeID", suffixes = c("_dalponte", "_watershed"))
cmp_ws$diam_diff_m <- cmp_ws$crown_diam_eq_m_watershed - cmp_ws$crown_diam_eq_m_dalponte
cat(sprintf("Median equivalent diameter - Dalponte: %.2f m | Watershed: %.2f m | median difference: %+.2f m
",
            median(cmp_ws$crown_diam_eq_m_dalponte), median(cmp_ws$crown_diam_eq_m_watershed), median(cmp_ws$diam_diff_m)))

# side-by-side whole-scene view
options(repr.plot.width = 20, repr.plot.height = 9)
par(mfrow = c(1, 2))
for (m in list(list("Dalponte", crowns), list("Marker-controlled watershed", crowns_ws))) {
  plot(chm_crown, range = c(0, ceiling(global(chm_crown, "max", na.rm = TRUE)[1, 1])), col = height.colors(50),
       legend = FALSE, main = m[[1]])
  plot(st_geometry(m[[2]]), border = "white", col = NA, lwd = 1, add = TRUE)
  points(trees$top_x, trees$top_y, pch = 3, col = "blue", cex = 0.6)
  points(pc[, 1], pc[, 2], pch = 4, col = "red", cex = 0.6)
}
par(mfrow = c(1, 1))

## 7. Crown Metrics

Area, equivalent diameter, and max Feret diameter from the crown polygons.

In [ ]:
crowns <- st_as_sf(as.polygons(crown_r, dissolve = TRUE))
crowns <- crowns[!is.na(crowns$treeID), ]

max_feret <- function(g) {
  h <- st_coordinates(st_convex_hull(g))[, 1:2]
  max(dist(h))
}
crowns$crown_area_m2     <- as.numeric(st_area(crowns))
crowns$crown_diam_eq_m   <- 2 * sqrt(crowns$crown_area_m2 / pi)
crowns$crown_diam_max_m  <- sapply(st_geometry(crowns), max_feret)

metrics <- st_drop_geometry(crowns)

# map tree IDs (merged seeds) back to every original point
trees <- trees %>%
  left_join(seeds %>% select(point_id, treeID), by = "point_id") %>%
  left_join(metrics, by = "treeID") %>%
  group_by(treeID) %>%
  mutate(n_points_in_crown = if_else(is.na(treeID), NA_integer_, n())) %>%
  ungroup()

summary(trees[, c("crown_area_m2", "crown_diam_eq_m", "crown_diam_max_m")])

## 8. Quality Control

- Height error vs. ground truth (`GT Height`)
- Points sharing a crown (`n_points_in_crown > 1`): the crown metrics belong to the group, not a single stem, so interpret with care
- Large `snap_dist_m` suggests the coordinate may be off or a taller neighbour captured the search window

In [ ]:
qc <- trees %>%
  mutate(gt = suppressWarnings(as.numeric(`GT Height`))) %>%
  filter(!is.na(gt), !is.na(chm_height_m)) %>%
  mutate(err = chm_height_m - gt)
cat("Trees with GT height:", nrow(qc), "\n")
cat(sprintf("Bias: %.2f m | MAE: %.2f m | RMSE: %.2f m | R2: %.3f\n",
            mean(qc$err), mean(abs(qc$err)), sqrt(mean(qc$err^2)), cor(qc$chm_height_m, qc$gt)^2))

cat("\nPoints sharing a crown:", sum(trees$n_points_in_crown > 1, na.rm = TRUE), "\n")
cat("Points with no crown assigned:", sum(is.na(trees$crown_area_m2)), "\n")
cat("Snap distance > 1.5 m:", sum(trees$snap_dist_m > 1.5, na.rm = TRUE), "\n")

plot(qc$gt, qc$chm_height_m, xlab = "Ground-truth height (m)", ylab = "CHM height (m)",
     pch = 19, col = rgb(0, 0, 0, 0.5)); abline(0, 1, col = "red")

## 9. Visualizations: Ground Coordinates vs. Drone Treetops vs. Crowns

Same convention as `tree_segmentation_dalponte.ipynb`: **red X** = ground-measured coordinate, **blue +** = drone treetop (snapped to the CHM maximum), **white solid outline** = Dalponte crown, **yellow dashed outline** = marker-controlled watershed crown (section 6b), black dashed line = snap offset.

In [ ]:
trees$label <- paste(trees$`Spec.`, trees$Tree, trees$Stem, sep = ".")
trees$gt_x <- pc[, 1]
trees$gt_y <- pc[, 2]
trees$crown_diam_eq_ws <- crowns_ws$crown_diam_eq_m[match(trees$treeID, crowns_ws$treeID)]
height_range <- c(0, ceiling(quantile(values(chm_crown), 0.995, na.rm = TRUE)))
col <- height.colors(50)

bbox_of <- function(e) st_bbox(c(xmin = xmin(e), xmax = xmax(e), ymin = ymin(e), ymax = ymax(e)), crs = st_crs(treetops))

In [ ]:
# Whole-scene overview
options(repr.plot.width = 14, repr.plot.height = 12)
plot(chm_crown, range = height_range, col = col, main = "All crowns, ground-measured points (red X) and drone treetops (blue +)")
plot(st_geometry(crowns), border = "white", col = NA, lwd = 1, add = TRUE)
plot(st_geometry(crowns_ws), border = "yellow", col = NA, lwd = 1, lty = 2, add = TRUE)
points(trees$top_x, trees$top_y, pch = 3, col = "blue", cex = 0.8, lwd = 2)
points(trees$gt_x, trees$gt_y, pch = 4, col = "red", cex = 0.8, lwd = 2)
legend("topright", legend = c("Ground-measured", "Drone treetop", "Dalponte crown", "Watershed crown"),
       pch = c(4, 3, NA, NA), lty = c(NA, NA, 1, 2), col = c("red", "blue", "white", "yellow"), lwd = 2, bg = "gray30", text.col = "white")

### Per-tree zoom panels: Dalponte vs. Watershed

Each surveyed point gets two panels side by side on the same window, centred on its ground coordinate: **left = Dalponte**, **right = marker-controlled watershed**. The highlighted outline is that tree's crown, other crowns in the window are faint. Titles give the crown's equivalent diameter and area.

In [ ]:
half <- 10            # half-width of each zoom window (m)
trees_per_page <- 8   # 4 x 4 grid = 8 trees x 2 panels
cmp_plot <- trees[!is.na(trees$top_x), ]

draw_tree_panel <- function(r, e, bb, crowns_m, ttl, outline) {
  plot(crop(chm_crown, e), col = col, range = height_range, legend = FALSE, main = ttl, cex.main = 0.85)
  cr_w <- crowns_m[st_intersects(crowns_m, st_as_sfc(bb), sparse = FALSE)[, 1], ]
  plot(st_geometry(cr_w), border = adjustcolor(outline, 0.5), col = NA, lwd = 1, add = TRUE)
  tt_w <- treetops[st_intersects(treetops, st_as_sfc(bb), sparse = FALSE)[, 1], ]
  plot(st_geometry(tt_w), add = TRUE, pch = 3, col = adjustcolor("blue", 0.35))
  if (!is.na(r$treeID)) plot(st_geometry(crowns_m[crowns_m$treeID == r$treeID, ]), border = outline, col = NA, lwd = 2.5, add = TRUE)
  segments(r$gt_x, r$gt_y, r$top_x, r$top_y, col = "black", lty = 2)
  points(r$top_x, r$top_y, pch = 3, col = "blue", lwd = 3, cex = 2)     # drone treetop
  points(r$gt_x, r$gt_y, pch = 4, col = "red", lwd = 3, cex = 2)        # ground-measured location
}

crown_txt <- function(d, a) ifelse(is.na(d), "no crown", sprintf("%.1f m dia | %.0f m2", d, a))

for (pg in split(seq_len(nrow(cmp_plot)), ceiling(seq_len(nrow(cmp_plot)) / trees_per_page))) {
  options(repr.plot.width = 16, repr.plot.height = 14)
  par(mfrow = c(4, 4), mar = c(2, 2, 3.5, 1))
  for (i in pg) {
    r  <- cmp_plot[i, ]
    e  <- ext(r$gt_x - half, r$gt_x + half, r$gt_y - half, r$gt_y + half)
    bb <- bbox_of(e)
    gt <- suppressWarnings(as.numeric(r$`GT Height`))
    nm <- ifelse(is.na(r$common_name) || !nzchar(r$common_name), "Unnamed species", r$common_name)
    ws_area <- crowns_ws$crown_area_m2[match(r$treeID, crowns_ws$treeID)]

    ttl_d <- sprintf("%s (%s)
GT %s vs CHM %.1f m | %.1f m apart
Dalponte: %s", nm, r$label,
                     ifelse(is.na(gt), "n/a", sprintf("%.1f m", gt)), r$chm_height_m, r$snap_dist_m,
                     crown_txt(r$crown_diam_eq_m, r$crown_area_m2))
    ttl_w <- sprintf("%s (%s)
GT %s vs CHM %.1f m | %.1f m apart
Watershed: %s", nm, r$label,
                     ifelse(is.na(gt), "n/a", sprintf("%.1f m", gt)), r$chm_height_m, r$snap_dist_m,
                     crown_txt(r$crown_diam_eq_ws, ws_area))

    draw_tree_panel(r, e, bb, crowns,    ttl_d, "white")
    draw_tree_panel(r, e, bb, crowns_ws, ttl_w, "yellow")
  }
  par(mfrow = c(1, 1))
}

### Per-footprint maps

All surveyed points in one footprint on a single map, labelled `Spec.Tree.Stem`.

In [ ]:
plot_footprint <- function(fp, pad = 8) {
  g <- trees[!is.na(trees$top_x) & trees$Footprint == fp, ]
  if (!nrow(g)) return(invisible(NULL))

  e  <- ext(min(g$gt_x, g$top_x) - pad, max(g$gt_x, g$top_x) + pad,
            min(g$gt_y, g$top_y) - pad, max(g$gt_y, g$top_y) + pad)
  bb <- bbox_of(e)

  options(repr.plot.width = 14, repr.plot.height = 12)
  plot(crop(chm_crown, e), col = col, range = height_range,
       main = sprintf("Footprint %s: ground-measured (red X) vs drone treetop (blue +); white = Dalponte, yellow dashed = watershed", fp))

  cr_w <- crowns[st_intersects(crowns, st_as_sfc(bb), sparse = FALSE)[, 1], ]
  plot(st_geometry(cr_w), border = adjustcolor("white", 0.5), col = NA, lwd = 1, add = TRUE)
  plot(st_geometry(crowns[crowns$treeID %in% g$treeID, ]), border = "white", col = NA, lwd = 2.5, add = TRUE)
  crw_w <- crowns_ws[st_intersects(crowns_ws, st_as_sfc(bb), sparse = FALSE)[, 1], ]
  plot(st_geometry(crw_w), border = adjustcolor("yellow", 0.5), col = NA, lwd = 1, lty = 2, add = TRUE)
  plot(st_geometry(crowns_ws[crowns_ws$treeID %in% g$treeID, ]), border = "yellow", col = NA, lwd = 2.5, lty = 2, add = TRUE)
  segments(g$gt_x, g$gt_y, g$top_x, g$top_y, col = "black", lwd = 1.5)
  points(g$top_x, g$top_y, pch = 3, col = "blue", lwd = 3, cex = 1.5)
  points(g$gt_x, g$gt_y, pch = 4, col = "red", lwd = 3, cex = 1.5)
  text(g$gt_x, g$gt_y, g$label, pos = 3, col = "white", cex = 1, font = 2)

  legend("topright", legend = c("Ground-measured", "Drone treetop", "Dalponte crown", "Watershed crown", "Snap offset"),
         pch = c(4, 3, NA, NA, NA), col = c("red", "blue", "white", "yellow", "black"),
         lty = c(NA, NA, 1, 2, 1), lwd = c(3, 3, 2.5, 2.5, 1.5), bg = "gray30", text.col = "white")
}

for (fp in sort(unique(na.omit(trees$Footprint)))) plot_footprint(fp)

## 10. Export

In [ ]:
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

out <- trees %>%
  mutate(GT_Height_m = suppressWarnings(as.numeric(`GT Height`))) %>%
  select(point_id, Spec = `Spec.`, Tree, Stem, Footprint, Lat, Long, top_x, top_y, snap_dist_m,
         chm_height_m, GT_Height_m, treeID, n_points_in_crown,
         crown_area_m2, crown_diam_eq_m, crown_diam_max_m)

write.csv(out, file.path(output_dir, "arboretum_tree_metrics_025m.csv"), row.names = FALSE)
st_write(treetops, file.path(output_dir, "arboretum_treetops_025m.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(crowns,   file.path(output_dir, "arboretum_crowns_025m.gpkg"),   delete_dsn = TRUE, quiet = TRUE)
cat("Wrote outputs to", output_dir, "\n")